In [ ]:
!pip install regex datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories", split = "train", streaming = True)

with open("/content/tiny_stories.txt", "w") as f:
  size = 0
  for ex in ds:
    f.write(ex["text"] + "\n")
    size += len(ex["text"])
    if size > 30_000_000:
      break

In [ ]:
text = "hello"
ids = list(text.encode("utf-8"))

In [ ]:
ids

[104, 101, 108, 108, 111, 32, 116, 104, 101, 114, 101]

In [ ]:
text = "a"
ids = list(text.encode("utf-8"))
ids

[97]

In [ ]:
def get_stats(ids):
  counts = {}
  for pair in zip(ids, ids[1:]):
    counts[pair] = counts.get(pair,0) + 1
  return counts

In [ ]:
def merge(ids, pair, new_id):
  out, i = [], 0
  while i < len(ids):
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      out.append(new_id)
      i += 2
    else:
      out.append(ids[i])
      i += 1
  return out


In [ ]:
merge([1, 2, 3, 1, 2], (1,2), 99)

[99, 3, 99]

In [ ]:
# the traoining loop
def train_simple(text, vocab_size):
  ids = list(text.encode("utf-8"))
  merges = {}
  for new_id in range(256, vocab_size):
    stats = get_stats(ids)
    pair = max(stats, key = stats.get)
    ids = merge(ids, pair, new_id)
    merges[pair] = new_id
  return merge

In [ ]:
def build_vocab(merges):
  vocab = {i : bytes([i]) for i in range(256)}
  for (a, b), new_id in merges.items():
    vocab[new_id] = vocab(a) + vocab[b]
  return vocab

In [ ]:
import regex as re, collections

# split into word-ish chunks so merges never cross word boundaries
pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

def merge(ids, pair, idx):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            out.append(idx); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

def train_bpe(text, vocab_size):
    counts = collections.Counter(pat.findall(text))
    words = {w: list(w.encode("utf-8")) for w in counts}
    merges = {}
    for idx in range(256, vocab_size):
        stats = collections.Counter()
        for w, ids in words.items():
            c = counts[w]
            for p in zip(ids, ids[1:]):
                stats[p] += c
        if not stats: break
        pair = max(stats, key=stats.get)
        merges[pair] = idx
        for w in words:
            words[w] = merge(words[w], pair, idx)
    return merges

def build_vocab(merges):
    vocab = {i: bytes([i]) for i in range(256)}
    for (a, b), idx in merges.items():   # dicts keep insertion order
        vocab[idx] = vocab[a] + vocab[b]
    return vocab

def encode(text, merges, cache):
    out = []
    for w in pat.findall(text):
        if w not in cache:
            ids = list(w.encode("utf-8"))
            while len(ids) >= 2:
                pairs = set(zip(ids, ids[1:]))
                pair = min(pairs, key=lambda p: merges.get(p, float("inf")))
                if pair not in merges: break
                ids = merge(ids, pair, merges[pair])
            cache[w] = ids
        out.extend(cache[w])
    return out

def decode(ids, vocab):
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")

In [ ]:
sample = open("/content/tiny_stories.txt").read()
merges = train_bpe(sample, vocab_size=4096)
vocab = build_vocab(merges)

cache = {}
ids = encode("Once upon a time, there was a little girl.", merges, cache)
assert decode(ids, vocab) == "Once upon a time, there was a little girl."

In [ ]:
print(len(merges))

3840


In [ ]:
import pickle
DRIVE_DIR = "/content/drive/MyDrive/projects_neural_practice"

with open(f"{DRIVE_DIR}/merges.pkl", "wb") as f:
    pickle.dump(merges, f)

In [ ]:
ids

[430, 442, 258, 395, 44, 397, 281, 258, 396, 447, 46]

In [ ]:
import numpy as np, pickle
from multiprocessing import Pool
from datasets import load_dataset

DRIVE_DIR = "/content/drive/MyDrive/projects_neural_practice"

merges = pickle.load(open(f"{DRIVE_DIR}/merges.pkl", "rb"))
EOT = 4096

cache = {}

def encode_story(text):
  return encode(text, merges, cache) + [EOT]

def tokenize_split(split, out_path, max_chars = None):
  ds = load_dataset("roneneldan/TinyStories", split = split, streaming = True)

  def stories():
    chars = 0
    for ex in ds:
      yield ex["text"]
      chars += len(ex["text"])
      if max_chars and chars >= max_chars:
        break

  total = 0
  with open(out_path, "wb") as f, Pool() as pool:
    for ids in pool.imap(encode_story, stories(), chunksize = 500):
      np.array(ids, dtype = np.uint16).tofile(f)
      total += len(ids)
  print(f"{split} : {total/1e6:.1f}M tokens -> {out_path}")

tokenize_split("validation", f"{DRIVE_DIR}/val.bin")
tokenize_split("train", f"{DRIVE_DIR}/train.bin", max_chars=1000_000_000)

validation : 4.9M tokens -> /content/drive/MyDrive/projects_neural_practice/val.bin
train : 255.9M tokens -> /content/drive/MyDrive/projects_neural_practice/train.bin


In [ ]:
!pwd

/content


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
DRIVE_DIR = "/content/drive/MyDrive/projects_neural_practice"

train = np.memmap(f"{DRIVE_DIR}/train.bin", dtype=np.uint16, mode="r")
val   = np.memmap(f"{DRIVE_DIR}/val.bin",   dtype=np.uint16, mode="r")

print(len(train), len(val))            # ~255.9M and ~4.9M
print(train.max(), val.max())          # both should be <= 4096
print((train[:5_000_000] == 4096).sum())   # number of stories in the first 5M tokens
print(decode(train[:300].tolist(), vocab)) # should read like a story

255925379 4886831
4096 4096
21943
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.<|endoftext|>Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree.

In [ ]:
vocab = build_vocab(merges)
vocab[4096] = b"<|endoftext|>"

In [ ]:
print(decode(train[:300].tolist(), vocab))

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.<|endoftext|>Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree had many leaves that wer